# ACRS v3 — integrated O1–O4 + G1 governance on a free GPU (Colab T4 / Kaggle)

**Jenisha T · M S Ramaiah University of Applied Sciences · 24ETRP720001**

`Runtime → Run all` executes, unattended:

1. the unit tests (fail-closed, kill switch, policy rules tested both ways, Learn-then-Test guarantee);
2. **ToolBench-Sessions G1**: objective ablation, prompt-injection sweep, certified execution, pass^k, release gate (BM25, then a GPU dense hybrid);
3. **KCC-Sessions**, the public problem: real Kisan Call Centre farmer queries, all four objectives integrated and governed.

Agents are simulated (seeded, schema-faithful), so the results test orchestration and governance, not LLM quality.

In [ ]:
import os, sys, json, time
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
WORK = '/kaggle/working' if IN_KAGGLE else '/content'
os.chdir(WORK)
print('platform:', 'Kaggle' if IN_KAGGLE else 'Colab/other')
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU (CPU parts still run)'

In [ ]:
BRANCH = 'acrs-v3-governed'
!rm -rf phd-agentic-ai-master && git clone -q --depth 1 -b {BRANCH} https://github.com/joyjeni/phd-agentic-ai-master.git
%cd phd-agentic-ai-master
!git log --oneline -1

In [ ]:
!pip -q install "sentence-transformers>=2.2" pyarrow huggingface_hub 2>&1 | tail -1
!cd laboratory/research && python -m pytest -q 2>&1 | tail -3

## 1 · ToolBench-Sessions (BM25 first stage, ~10 min)

In [ ]:
!python experiments/integrated_eval.py --dataset toolbench --seeds 3 --out experiments/results/integrated_toolbench.json 2>&1 | grep -E '^\[|wrote'

## 2 · ToolBench-Sessions with a GPU dense encoder (hybrid BM25 + bge-base-en-v1.5)

In [ ]:
os.environ['ACRS_DENSE_MODEL'] = 'BAAI/bge-base-en-v1.5'
import torch; print('cuda:', torch.cuda.is_available())
!python experiments/integrated_eval.py --dataset toolbench --seeds 3 --retriever hybrid --quick --attack-rates 0,0.15 --out experiments/results/integrated_toolbench_hybrid_bge.json 2>&1 | grep -E '^\[|wrote'

## 3 · KCC-Sessions — the public problem

Source: Kisan Call Centre transcripts (ICAR-IASRI KCC Chakshu / data.gov.in / IndiaAI AIKosh). Download route: the 2023 parquet mirror on Hugging Face (`Omegaindebt/Kisan_Call_Centre_Transcripts`, 1.0M rows, original 11-field schema). We draw a seeded **stratified sample over QueryType** and keep whole block-day queues, so pseudo-sessions stay multi-turn.

In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd
p = hf_hub_download('Omegaindebt/Kisan_Call_Centre_Transcripts', 'default/train/0000.parquet',
                    repo_type='dataset', revision='refs/convert/parquet')
df = pd.read_parquet(p)
print(df.shape); print(df['QueryType'].astype(str).str.strip().str.lower().value_counts().head(25))

In [ ]:
sys.path.insert(0, 'laboratory/research/src')
from paim.data.kcc import QUERY_TYPE_TOOLS, normalise_query_type
df['qt'] = df['QueryType'].astype(str).map(normalise_query_type)
keep = df['qt'].isin(QUERY_TYPE_TOOLS) & df['QueryText'].notna() & (df['Sector'].astype(str).str.upper() == 'AGRICULTURE')
print('mappable rows:', int(keep.sum()), 'of', len(df))
df = df[keep].copy()
df['day'] = df['CreatedOn'].astype(str).str[:10]
PER_TYPE = 250
seed_rows = df.groupby('qt', group_keys=False).apply(lambda g: g.sample(min(len(g), PER_TYPE), random_state=20260905))
q = seed_rows[['StateName', 'DistrictName', 'BlockName', 'day']].drop_duplicates()
queues = df.merge(q, on=['StateName', 'DistrictName', 'BlockName', 'day']).drop(columns=['qt', 'day']).drop_duplicates()
os.makedirs('data_kcc', exist_ok=True)
queues.to_csv('data_kcc/kcc_sample.csv', index=False)
print('sample rows written:', len(queues))

In [ ]:
!python experiments/integrated_eval.py --dataset kcc --kcc data_kcc/kcc_sample.csv --limit 0 --distractors --seeds 3 --attack-rates 0,0.15 --out experiments/results/integrated_kcc.json 2>&1 | grep -E '^\[|wrote|dataset'

## 4 · Summary tables

In [ ]:
KEYS = ['ndcg@5','executeRate','executedErrorRate','executedPrecision','attackSuccessRate','costPerSubGoal','compression','fidelityFloorMet','stageFailureRate']
def table(R, section):
    return pd.DataFrame([{'config': n, **{k: round(d['dist'][k]['median'], 4) for k in KEYS if k in d['dist']}}
                         for n, d in R.get(section, {}).items()])
for f in ['integrated_toolbench.json', 'integrated_toolbench_hybrid_bge.json', 'integrated_kcc.json']:
    path = f'experiments/results/{f}'
    if not os.path.exists(path):
        continue
    R = json.load(open(path)); print('\n====', f, '| runtime s:', R['runtimeSeconds'])
    for sec in ['ablation', 'attack']:
        t = table(R, sec)
        if len(t): display(t)
    display(pd.DataFrame([{'alpha': c['alpha'], 'seed': c['seed'], 'attack': c['attack'],
        'theta*': c['threshold']['threshold'], 'valid': c['threshold']['valid'],
        'UCB': round(c['threshold']['risk_upper'], 4), 'heldOutRisk': round(c['heldOutExecutedRisk'], 4),
        'coverage': round(c['heldOutCoverage'], 3), 'withinAlpha': c['riskWithinAlpha'],
        'ECE': round(c['eceWeightedShare'], 3)} for c in R['certified']]))
    print('pass^k', R['passK']); print('release gate', R['release']['gate'], '| CI may promote:', R['release']['ciMayPromote'])

In [ ]:
import shutil
shutil.make_archive(f'{WORK}/acrs_v3_results', 'zip', 'experiments/results')
print('download:', f'{WORK}/acrs_v3_results.zip')